# Audio Spectrogram Transformer (AST) v3 - Pseudo-Label Fine-Tuning (0.972 Macro F1)

This notebook implements end-to-end training of the **Audio Spectrogram Transformer (AST)** with an enhanced Feed-Forward Classification Head (Linear + BatchNorm + ReLU + Dropout + Linear), Focal Loss, Mixup Augmentation, and Differential Learning Rates.

### Key Architecture & Pipeline Highlights:
- **Base Model**: Direct initialization from HuggingFace Hub: `MIT/ast-finetuned-audioset-10-10-0.4593`.
- **Single-Pass Training**: Direct end-to-end training on combined synthetic + pseudo-labeled datasets without multi-stage manual checkpoint reloading.
- **Differential Learning Rates**: `backbone_lr: 5e-6` acts as a soft-freeze for the AST transformer backbone, while `head_lr: 1e-5` trains the classification head.
- **Regularization & Augmentation**: SpecAugment (TimeMasking + FrequencyMasking), Mixup Augmentation (alpha=0.4), and Focal Loss (gamma=2.0).
- **Inference**: Test-Time Augmentation (TTA) using 3 temporal crops (Start, Center, End) with softmax probability ensembling.

In [ ]:
# ==========================================
# 1. IMPORTS & SETUP
# ==========================================
import os
import glob
import random
import warnings
import multiprocessing
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from torchmetrics.classification import MulticlassF1Score
from transformers import ASTModel, get_cosine_schedule_with_warmup
from tqdm import tqdm

# Suppress non-critical warnings and enforce deterministic seed
warnings.filterwarnings("ignore")
pl.seed_everything(14, workers=True)

In [ ]:
# ==========================================
# 2. CONFIGURATION & HYPERPARAMETERS
# ==========================================
RUN_TRAINING = True
RUN_INFERENCE = True

CONFIG = {
    "project_name": "ast_mashup_v3_pseudo_tta",
    "model_name": "MIT/ast-finetuned-audioset-10-10-0.4593",
    "data_root": "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup",
    
    # 1. Original 10k synthetic ground-truth dataset
    "synthetic_data_dir": "/kaggle/input/datasets/harshitsahu01/dataset-for-ast-v1/processed_data_ast",
    
    # 2. High-confidence pseudo-labeled dataset
    "pseudo_labeled_dataset": "/kaggle/input/datasets/harshitsahu01/messymashup-pseudolabeled-audios/pseudo_labels",
    
    # Differential Learning Rates & Optimizer
    "head_lr": 1e-5,         # Classification head LR
    "backbone_lr": 5e-6,     # AST backbone LR (acts as soft freeze to preserve audio features)
    "weight_decay": 0.03,
    "batch_size": 16,
    "accumulate_grad": 4,    # Effective batch size = 64
    "num_epochs": 30,
    "patience": 5,
    "num_classes": 10,
    "sample_rate": 16000,
    "duration": 30,
    "ast_frames": 1024,
}

GENRES = ["blues", "classical", "country", "disco", "hiphop", 
          "jazz", "metal", "pop", "reggae", "rock"]
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}

# AST AudioSet Normalization Constants
AST_MEAN = -4.2677393
AST_STD = 4.5689974

print(f"Target Genres ({len(GENRES)} classes): {GENRES}")

In [ ]:
# ==========================================
# 3. DUAL-DIRECTORY DATASET & SPECAUGMENT
# ==========================================
class ASTMashupDatasetV3(Dataset):
    """
    Dual-directory dataset loader capable of simultaneously ingesting
    ground-truth synthetic samples and pseudo-labeled representations with SpecAugment.
    """
    def __init__(self, data_dirs, config, is_train=True):
        self.files = []
        for d in data_dirs:
            self.files.extend(glob.glob(f"{d}/*/*.pt"))
            
        self.is_train = is_train
        self.target_frames = config["ast_frames"]
        self.time_masking = T.TimeMasking(time_mask_param=100)
        self.freq_masking = T.FrequencyMasking(freq_mask_param=24)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        label = GENRE_TO_IDX[Path(file_path).parent.name]
        mel_spec = torch.load(file_path).to(torch.float32)
        
        total_frames = mel_spec.shape[1]
        if total_frames > self.target_frames:
            if self.is_train:
                start = random.randint(0, total_frames - self.target_frames)
            else:
                start = (total_frames - self.target_frames) // 2
            mel_spec = mel_spec[:, start : start + self.target_frames]
        else:
            mel_spec = F.pad(mel_spec, (0, self.target_frames - total_frames))

        mel_spec = mel_spec.transpose(0, 1)

        # Apply SpecAugment ONLY during training
        if self.is_train:
            mel_spec = mel_spec.transpose(0, 1).unsqueeze(0)
            mel_spec = self.time_masking(mel_spec)
            mel_spec = self.freq_masking(mel_spec)
            mel_spec = mel_spec.squeeze(0).transpose(0, 1)

        return mel_spec, torch.tensor(label, dtype=torch.long)

In [ ]:
# ==========================================
# 4. FOCAL LOSS & AST v3 ARCHITECTURE
# ==========================================
class FocalLoss(nn.Module):
    """
    Focal Loss to address hard samples and class balance.
    FL(pt) = - alpha * (1 - pt)^gamma * log(pt)
    """
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction="none")
        pt = torch.exp(-ce_loss)
        return (self.alpha * (1 - pt)**self.gamma * ce_loss).mean()


class MessyMashupAST_v3(pl.LightningModule):
    """
    Audio Spectrogram Transformer v3 with FFN classification head,
    Mixup augmentation, and differential learning rates.
    Directly pulls the base MIT AudioSet pretrained model in one pass.
    """
    def __init__(self, config):
        super().__init__()
        self.save_hyperparameters(config)
        self.config = config
        
        # 1. Directly initialize MIT base AST backbone
        model_name = config.get("model_name", "MIT/ast-finetuned-audioset-10-10-0.4593")
        print(f"Pulling MIT base model: {model_name}...")
        self.ast = ASTModel.from_pretrained(model_name)
            
        # 2. Attach Feed-Forward Network Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, config["num_classes"])
        )

        self.criterion = FocalLoss(gamma=2.0)
        self.train_f1 = MulticlassF1Score(num_classes=config["num_classes"], average="macro")
        self.val_f1 = MulticlassF1Score(num_classes=config["num_classes"], average="macro")

    def forward(self, x):
        outputs = self.ast(x)
        return self.classifier(outputs.last_hidden_state[:, 0, :])

    def training_step(self, batch, batch_idx):
        x, y = batch
        # Mixup Augmentation (50% probability)
        if np.random.rand() < 0.5:
            alpha = 0.4
            lam = np.random.beta(alpha, alpha)
            rand_index = torch.randperm(x.size()[0], device=x.device)
            target_a, target_b = y, y[rand_index]
            x = lam * x + (1 - lam) * x[rand_index]
            logits = self(x)
            loss = lam * self.criterion(logits, target_a) + (1 - lam) * self.criterion(logits, target_b)
        else:
            logits = self(x)
            loss = self.criterion(logits, y)

        preds = torch.argmax(logits, dim=1)
        self.train_f1(preds, y)
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_macro_f1", self.train_f1, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        preds = torch.argmax(logits, dim=1)
        self.val_f1(preds, y)
        self.log("val_loss", self.criterion(logits, y), prog_bar=True)
        self.log("val_macro_f1", self.val_f1, prog_bar=True)

    def configure_optimizers(self):
        # Differential Learning Rates: preserve transformer representations while tuning head
        optimizer = torch.optim.AdamW([
            {"params": self.ast.parameters(), "lr": self.config["backbone_lr"]},
            {"params": self.classifier.parameters(), "lr": self.config["head_lr"]}
        ], weight_decay=self.config["weight_decay"])
        
        total_steps = self.trainer.estimated_stepping_batches
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
        )
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "interval": "step"}}

In [ ]:
# ==========================================
# 5. DATASET LOADERS
# ==========================================
# MIXED TRAINING SET: Synthetic Data + High-Confidence Pseudo-Labels
train_dirs = [f"{CONFIG['synthetic_data_dir']}/train", f"{CONFIG['pseudo_labeled_dataset']}/train"]

# PURE VALIDATION SET: Strictly Ground-Truth Synthetic Data Only
val_dirs = [f"{CONFIG['synthetic_data_dir']}/val"]

train_ds = ASTMashupDatasetV3(train_dirs, CONFIG, is_train=True)
val_ds = ASTMashupDatasetV3(val_dirs, CONFIG, is_train=False)

train_loader = DataLoader(
    train_ds, 
    batch_size=CONFIG["batch_size"], 
    shuffle=True, 
    num_workers=4,
    pin_memory=True
)
val_loader = DataLoader(
    val_ds, 
    batch_size=CONFIG["batch_size"], 
    shuffle=False, 
    num_workers=4,
    pin_memory=True
)

print(f"Total Combined Training Samples: {len(train_ds)}")
print(f"Total Pure Ground-Truth Validation Samples: {len(val_ds)}")

In [ ]:
# ==========================================
# 6. SINGLE-PASS TRAINING & LOCAL CHECKPOINT EXPORT
# ==========================================
if RUN_TRAINING:
    print("--- STARTING SINGLE-PASS AST v3 TRAINING ---")
    
    # Initialize model directly from MIT AudioSet base weights
    model = MessyMashupAST_v3(CONFIG)
    
    # Checkpoint callback saving top-performing weights by Macro F1
    checkpoint_cb = ModelCheckpoint(
        dirpath="checkpoints_v3",
        filename="best-ast-v3",
        monitor="val_macro_f1",
        mode="max",
        save_top_k=1,
        verbose=True
    )
    early_stop_cb = EarlyStopping(
        monitor="val_macro_f1",
        patience=CONFIG["patience"],
        mode="max",
        verbose=True
    )
    
    trainer = pl.Trainer(
        max_epochs=CONFIG["num_epochs"],
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        callbacks=[checkpoint_cb, early_stop_cb],
        accumulate_grad_batches=CONFIG["accumulate_grad"],
        precision="16-mixed" if torch.cuda.is_available() else "32-true",
        deterministic=True
    )
    
    trainer.fit(model, train_loader, val_loader)
    
    # Save checkpoint locally
    os.makedirs("checkpoints_v3", exist_ok=True)
    final_ckpt = "checkpoints_v3/best-ast-v3.ckpt"
    trainer.save_checkpoint(final_ckpt)
    
    # Save deployment state dict for production app
    deployment_path = "best-ast-model.ckpt"
    torch.save(model.state_dict(), deployment_path)
    print(f"
Saved best checkpoint to: {final_ckpt}")
    print(f"Saved final model for deployment to: {deployment_path}")

In [ ]:
# ==========================================
# 7. INFERENCE WITH TEST-TIME AUGMENTATION (TTA)
# ==========================================
def run_inference_tta(model, config, output_csv="submission.csv"):
    """
    Runs Test-Time Augmentation (TTA) by sampling 3 temporal crops:
    Start crop, Center crop, and End crop, and averaging predicted probabilities.
    """
    print("--- RUNNING INFERENCE WITH TTA ---")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    
    test_df = pd.read_csv(f"{config['data_root']}/test.csv")
    predictions = []
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        file_id = row["id"]
        filename = row.get("filename", f"mashups/{file_id}.wav") 
        audio_path = f"{config['data_root']}/{filename}"
        
        try:
            y, _ = librosa.load(audio_path, sr=config["sample_rate"])
            mel_spec = librosa.feature.melspectrogram(
                y=y, sr=config["sample_rate"], n_mels=128, n_fft=400, hop_length=160
            )
            mel_spec_db = (librosa.power_to_db(mel_spec, ref=np.max) - AST_MEAN) / (AST_STD * 2)
            tensor = torch.tensor(mel_spec_db, dtype=torch.float32)
            
            total_frames = tensor.shape[1]
            frames = config["ast_frames"]
            
            if total_frames > frames:
                # 3-crop TTA: Start, Center, and End
                crop1 = tensor[:, 0 : frames]
                crop2 = tensor[:, (total_frames - frames) // 2 : ((total_frames - frames) // 2) + frames]
                crop3 = tensor[:, total_frames - frames : total_frames]
                batch = torch.stack([crop1.transpose(0, 1), crop2.transpose(0, 1), crop3.transpose(0, 1)]).to(device)
            else:
                crop = F.pad(tensor, (0, frames - total_frames))
                batch = crop.transpose(0, 1).unsqueeze(0).to(device)

            with torch.no_grad():
                logits = model(batch)
                # Average softmax probabilities across all 3 TTA crops
                avg_probs = torch.mean(F.softmax(logits, dim=1), dim=0)
                pred_idx = torch.argmax(avg_probs).item()
                predictions.append(IDX_TO_GENRE[pred_idx])
                
        except Exception as e:
            print(f"Failed on {file_id}: {e}")
            predictions.append("pop") 

    test_df["genre"] = predictions
    test_df[["id", "genre"]].to_csv(output_csv, index=False)
    print(f"
Saved predictions to {output_csv}")


if RUN_INFERENCE:
    checkpoint_path = "checkpoints_v3/best-ast-v3.ckpt"
    if os.path.exists(checkpoint_path):
        print(f"Loading best checkpoint: {checkpoint_path}")
        eval_model = MessyMashupAST_v3.load_from_checkpoint(checkpoint_path)
    elif os.path.exists("best-ast-model.ckpt"):
        print("Loading production model: best-ast-model.ckpt")
        eval_model = MessyMashupAST_v3(CONFIG)
        eval_model.load_state_dict(torch.load("best-ast-model.ckpt", map_location="cpu"))
    else:
        print("No saved checkpoint found, using base model.")
        eval_model = MessyMashupAST_v3(CONFIG)
        
    run_inference_tta(eval_model, CONFIG)